# Pretrained Encoder + JacobianODE — Lorenz (Evaluation)

Load a trained **Pretrained Encoder + JacobianODE** model from a W&B run
and compute evaluation diagnostics:

1. Observation-space prediction quality
2. Per-timestep prediction loss
3. Prediction visualisation
4. Latent space PCA
5. Lyapunov exponent comparison (estimated vs analytical vs theoretical)
6. Estimated dimension & participation ratio
7. Latent utilisation & S_dim
8. Jacobian consistency (velocity propagation)
9. False Nearest Neighbors (FNN) loss
10. Noise amplification loss

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import matplotlib.pyplot as plt
import numpy as np
import torch

from JacobianODE.encoder_only.pretrained import load_pretrained_jac_run
from JacobianODE.models.latent_jacobian import LitLatentJacobianODE
from JacobianODE.fnn import compute_variances, compute_s_dim
from JacobianODE.jacobians.metrics import r2_score

torch.set_float32_matmul_precision('high')

## 1. Load Run from W&B

Reconstructs the full pipeline (encoder adapter, Jacobian MLP, data) from
the W&B run config and loads the best checkpoint.

In [3]:
# ----------------------------------------------------------------
# W&B run to evaluate (EDIT THESE)
# ----------------------------------------------------------------
PROJECT  = "Lorenz__PretrainedEncoderJacODE"
RUN_ID   = "efcwbhz9"
SAVE_DIR = "/orcd/data/ekmiller/001/eisenaj/JacobianODE/lightning/pretrained_jac_runs"

# Known Lorenz Lyapunov exponents (sigma=10, rho=28, beta=8/3)
TRUE_LYAPUNOV = [0.91, 0.0, -14.57]

In [5]:
result = load_pretrained_jac_run(
    project=PROJECT,
    run_id=RUN_ID,
    save_dir=SAVE_DIR,
    verbose=True,
)

# Unpack for convenience
run           = result.run
cfg           = result.cfg
eq            = result.eq
dt            = result.dt
values        = result.values
mu            = result.mu
sigma         = result.sigma
noise_scale_factor = result.noise_scale_factor
train_dl      = result.train_dl
val_dl        = result.val_dl
test_dl       = result.test_dl
trajs         = result.trajs
test_trajs_full = result.test_trajs_full
adapter       = result.adapter
lit_model     = result.lit_model
n_obs         = result.n_obs

N_LATENT = adapter.n_latent
lit_model.true_lyapunov_exponents = torch.tensor(TRUE_LYAPUNOV, dtype=torch.float32)

print(f"\nRun: {run.name}  (id={run.id})")
print(f"Encoder: {type(adapter.encoder).__name__}, n_latent={N_LATENT}")
print(f"n_obs={n_obs}, dt={dt:.4f}")
if test_trajs_full is not None:
    print(f"test_trajs_full: {test_trajs_full.shape}")

Loaded config from run Lorenz__SSMSequenceEncoder__n10__frozen__lc0.0__pred10 (id=efcwbhz9)


Sequence Indices:   0%|          | 0/1058 [00:00<?, ?it/s]

Train dataset shape: torch.Size([23276, 100, 43])
Validation dataset shape: torch.Size([7406, 100, 43])
Test dataset shape: torch.Size([3174, 100, 43])
Train trajectories dataset shape: torch.Size([22, 1158, 43])
Validation trajectories dataset shape: torch.Size([7, 1158, 43])
Test trajectories dataset shape: torch.Size([3, 1158, 43])
n_obs = 43, test_trajs_full = torch.Size([3, 1200, 3])
Encoder: SSMSequenceEncoder, d_model=128, n_latent=10, n_layers=3, context_margin=0
Model loaded: 1,221,401 params (1,221,401 trainable)

Run: Lorenz__SSMSequenceEncoder__n10__frozen__lc0.0__pred10  (id=efcwbhz9)
Encoder: SSMSequenceEncoder, n_latent=10
n_obs=43, dt=0.0150
test_trajs_full: torch.Size([3, 1200, 3])


## 2. Encode Full Test Trajectories + Compute Jacobians

In [6]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
lit_model = lit_model.to(device)
lit_model.eval()

test_trajs_seq = trajs['test_trajs'].sequence

with torch.no_grad():
    latent_traj_full = lit_model.encode_trajectory(test_trajs_seq.to(device))
    traj_decoded_full = lit_model.decode_trajectory(latent_traj_full)
    jacs_full = lit_model.compute_jacobians(latent_traj_full)
    lyaps_full = lit_model.compute_lyapunov_exponents(jacs_full, dt)

latent_traj_full = latent_traj_full.cpu()
traj_decoded_full = traj_decoded_full.cpu()
jacs_full = jacs_full.cpu()
lyaps_full = lyaps_full.cpu()
lit_model = lit_model.cpu()

print(f"Latent trajectories:  {latent_traj_full.shape}")
print(f"Decoded trajectories: {traj_decoded_full.shape}")
print(f"Jacobians:            {jacs_full.shape}")
print(f"\nEstimated Lyapunov exponents (per trajectory):")
print(f"  {lyaps_full.numpy()}")
print(f"  mean: {lyaps_full.mean(0).numpy()}")
print(f"\nTrue Lorenz: {TRUE_LYAPUNOV}")

Latent trajectories:  torch.Size([3, 1158, 10])
Decoded trajectories: torch.Size([3, 1158, 43])
Jacobians:            torch.Size([3, 1158, 10, 10])

Estimated Lyapunov exponents (per trajectory):
  [[ -0.13661054  -0.42910042  -2.2222352   -2.2475948   -7.395496
   -8.302187    -8.894251    -9.17859    -12.894384   -13.03699   ]
 [ -0.22923766  -0.3637443   -2.175878    -2.2248976   -7.4287353
   -8.267375    -8.933127    -9.181568   -12.923276   -13.032433  ]
 [  0.15390761  -0.14656009  -2.2834368   -3.684347    -3.916429
   -7.165888    -7.257909    -7.603887    -9.456044   -12.003271  ]]
  mean: [ -0.07064686  -0.31313494  -2.2271833   -2.7189465   -6.2468867
  -7.911816    -8.361762    -8.654681   -11.757901   -12.690898  ]

True Lorenz: [0.91, 0.0, -14.57]


## 3. Observation-Space Prediction Quality

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
lit_model = lit_model.to(device)
lit_model.eval()

val_losses = []
all_z = []
all_obs = []

with torch.no_grad():
    for batch in val_dl:
        if isinstance(batch, (list, tuple)):
            batch = batch[0]
        batch = batch.to(device)

        ret = lit_model.trajectory_model_step(
            batch, alpha_teacher_forcing=0, obs_noise_scale=0,
        )
        val_losses.append(ret['loss'].item())

        z = lit_model.encode_trajectory(batch)
        all_z.append(z.cpu())
        all_obs.append(batch.cpu())

lit_model = lit_model.cpu()
print(f"Validation trajectory loss (no teacher forcing): {np.mean(val_losses):.6f}")

z_all = torch.cat(all_z, dim=0)
obs_all = torch.cat(all_obs, dim=0)

## 4. Per-Timestep Prediction Loss

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
lit_model_dev = lit_model.to(device)
lit_model_dev.eval()

PREDICTION_STEPS = int(cfg.model.prediction_steps)
TRAJ_INIT_STEPS = int(lit_model_dev.jacobianODEint_kwargs.get('traj_init_steps', 15))

orig_pred_steps = lit_model_dev.prediction_steps
lit_model_dev.prediction_steps = 100

with torch.no_grad():
    traj_ret = lit_model_dev.trajectory_model_step(
        test_trajs_seq.to(device), alpha_teacher_forcing=0.0, return_decoded=True,
    )
    targets = traj_ret['targets']
    decoded = traj_ret['decoded']
    z_pred_full = traj_ret['outputs']
    latent_targets = lit_model_dev.encode_trajectory(targets)

per_step_latent = ((latent_targets - z_pred_full[:, TRAJ_INIT_STEPS:])**2).mean(dim=(0, 2)).cpu().numpy()
per_step_obs = ((targets.cpu() - decoded.cpu())**2).mean(dim=(0, 2)).numpy()

ten_step_latent = per_step_latent[:10].mean()
ten_step_obs = per_step_obs[:10].mean()

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(per_step_latent / ten_step_latent, label='Latent loss (normalized)')
ax.plot(per_step_obs / ten_step_obs, label='Observation loss (normalized)')
ax.axhline(1.0, color='k', linestyle='--', lw=0.5, label='1.0')
ax.set_xscale('log')
ax.set_yscale('log')
ax.set_xlabel('Prediction step')
ax.set_ylabel('Normalized MSE')
ax.set_title('Per-Timestep Prediction Loss (100-step horizon)')
ax.legend()
plt.tight_layout()
plt.show()

lit_model_dev.prediction_steps = orig_pred_steps
lit_model = lit_model_dev.cpu()

## 5. Prediction Visualisation

In [ ]:
INTERP_PTS = int(lit_model.jacobianODEint_kwargs.get('interp_pts', 4))
INNER_N = int(lit_model.jacobianODEint_kwargs.get('inner_N', 20))

lit_model.eval()
test_batch = next(iter(test_dl))
if isinstance(test_batch, (list, tuple)):
    test_batch = test_batch[0]

with torch.no_grad():
    z_test = lit_model.encode_trajectory(test_batch)
    window_len = TRAJ_INIT_STEPS + PREDICTION_STEPS
    z_window = z_test[:, :window_len, :]

    from JacobianODE.jacobians.jacobianODE import JacobianODEint
    jac_odeint = JacobianODEint(lit_model.compute_jacobians, dt)
    z_pred = jac_odeint.generate_dynamics(
        z_window,
        alpha_teacher_forcing=0,
        fast_mode=True,
        traj_init_steps=TRAJ_INIT_STEPS,
        interp_pts=INTERP_PTS,
        inner_N=INNER_N,
        inner_path='line',
    )

    z_pred_crop = z_pred[:, TRAJ_INIT_STEPS:, :]
    z_true_crop = z_window[:, TRAJ_INIT_STEPS:, :]
    decoded_pred = lit_model.decode_trajectory(z_pred_crop)
    decoded_true = lit_model.decode_trajectory(z_true_crop)

n_show = min(4, decoded_pred.shape[0])
fig, axes = plt.subplots(n_show, 1, figsize=(12, 3 * n_show), sharex=True)
if n_show == 1:
    axes = [axes]

for i in range(n_show):
    pred_signal = decoded_pred[i].reshape(-1).numpy()
    true_signal = decoded_true[i].reshape(-1).numpy()
    axes[i].plot(true_signal, label='True (decoded from true z)', alpha=0.8)
    axes[i].plot(pred_signal, '--', label='Predicted (decoded from JacODE z)', alpha=0.8)
    axes[i].set_ylabel('x(t)')
    axes[i].set_title(f'Batch element {i}')
    if i == 0:
        axes[i].legend()

axes[-1].set_xlabel('Time index')
fig.suptitle(f'Obs-Space Predictions ({PREDICTION_STEPS} latent steps forward)', y=1.01)
plt.tight_layout()
plt.show()

## 6. Latent Space Visualisation (PCA)

In [ ]:
from sklearn.decomposition import PCA

z_flat = z_all.reshape(-1, z_all.shape[-1]).numpy()
pca = PCA(n_components=3)
z_pca = pca.fit_transform(z_flat)

fig = plt.figure(figsize=(14, 5))

ax1 = fig.add_subplot(131, projection='3d')
n_plot = min(5000, len(z_pca))
idx = np.random.choice(len(z_pca), n_plot, replace=False)
ax1.scatter(z_pca[idx, 0], z_pca[idx, 1], z_pca[idx, 2], s=0.5, alpha=0.3)
ax1.set_title('Latent Space (PCA)')
ax1.set_xlabel('PC1'); ax1.set_ylabel('PC2'); ax1.set_zlabel('PC3')

ax2 = fig.add_subplot(132)
ax2.scatter(z_pca[idx, 0], z_pca[idx, 1], s=0.5, alpha=0.3)
ax2.set_xlabel('PC1'); ax2.set_ylabel('PC2')
ax2.set_title('Latent PC1 vs PC2')

ax3 = fig.add_subplot(133)
ax3.bar(range(1, 4), pca.explained_variance_ratio_)
ax3.set_xlabel('Component'); ax3.set_ylabel('Variance Explained')
ax3.set_title(f'PCA Variance ({pca.explained_variance_ratio_.sum():.1%} total)')

plt.tight_layout()
plt.show()

## 7. Lyapunov Exponent Comparison

Three-way comparison:
- **Estimated**: from learned latent Jacobians (QR decomposition)
- **Analytical**: from the true Lorenz Jacobian on full 3-D test trajectories
- **Theoretical**: known values for the Lorenz attractor

In [ ]:
est_mean = lyaps_full.mean(0).numpy()

# Compute true Lyapunov from analytical Lorenz Jacobians (if full obs available)
lyaps_true = None
if test_trajs_full is not None:
    def lorenz_jacobian(x_raw, sigma_lz=10.0, rho=28.0, beta=8.0/3.0):
        x1, x2, x3 = float(x_raw[0]), float(x_raw[1]), float(x_raw[2])
        return torch.tensor([
            [-sigma_lz, sigma_lz,    0.0 ],
            [rho - x3,  -1.0,       -x1  ],
            [x2,         x1,        -beta],
        ], dtype=torch.float32)

    sigma_t = torch.as_tensor(sigma, dtype=torch.float32)
    mu_t    = torch.as_tensor(mu,    dtype=torch.float32)

    B, T, D = test_trajs_full.shape
    true_jacs = torch.zeros(B, T, D, D, dtype=torch.float32)
    for b in range(B):
        for t in range(T):
            x_raw = test_trajs_full[b, t] * sigma_t + mu_t
            true_jacs[b, t] = lorenz_jacobian(x_raw)

    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    with torch.no_grad():
        lyaps_true = lit_model.to(device).compute_lyapunov_exponents(
            true_jacs.to(device), dt
        ).cpu()
    lit_model = lit_model.cpu()

    true_mean = lyaps_true.mean(0).numpy()
    print('True Lyapunov exponents  (analytical Lorenz Jacobians, full 3-D test trajs)')
    print(f'  per trajectory : {lyaps_true.numpy()}')
    print(f'  mean           : {true_mean}')
    print()

print('Estimated Lyapunov exponents  (learned latent Jacobians)')
print(f'  per trajectory : {lyaps_full.numpy()}')
print(f'  mean           : {est_mean}')
print(f'\nKnown theoretical values : {TRUE_LYAPUNOV}')

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
n_est = len(est_mean)
n_known = len(TRUE_LYAPUNOV)
width = 0.25

ax.bar(np.arange(n_est) - width, est_mean, width, label='Estimated (latent J)', alpha=0.8, color='C0')
if lyaps_true is not None:
    true_mean = lyaps_true.mean(0).numpy()
    ax.bar(np.arange(len(true_mean)), true_mean, width, label='Analytical (Lorenz J)', alpha=0.8, color='C1')
ax.bar(np.arange(n_known) + width, TRUE_LYAPUNOV, width, label='Theoretical', alpha=0.8, color='C2')
ax.axhline(y=0, color='k', linestyle='--', lw=0.5)
ax.set_xlabel('Exponent index')
ax.set_ylabel('Lyapunov exponent')
ax.set_title('Lyapunov Spectrum: Estimated vs Analytical vs Theoretical')
ax.legend()
plt.tight_layout()
plt.show()

## 8. Estimated Dimension & Participation Ratio

In [ ]:
latent_vars = latent_traj_full.reshape(-1, latent_traj_full.shape[-1]).var(dim=0)
est_dim_latent = (latent_vars / latent_vars.max()).sum()

def participation_ratio(X):
    X_flat = X.reshape(-1, X.shape[-1])
    if hasattr(X_flat, 'numpy'):
        X_flat = X_flat.numpy()
    cov = np.cov(X_flat, rowvar=False)
    eigvals = np.linalg.eigvalsh(cov)
    return (eigvals.sum())**2 / (eigvals**2).sum()

pr_latent = participation_ratio(latent_traj_full)

if test_trajs_full is not None:
    true_vars = test_trajs_full.reshape(-1, test_trajs_full.shape[-1]).var(axis=0)
    est_dim_true = (true_vars / true_vars.max()).sum()
    pr_true = participation_ratio(test_trajs_full)

    labels = ['Estimated Dimension', 'Participation Ratio']
    true_values = [float(est_dim_true), pr_true]
    latent_values = [est_dim_latent.item(), pr_latent]

    x = np.arange(len(labels))
    width = 0.35
    fig, ax = plt.subplots(figsize=(6, 5))
    bars1 = ax.bar(x - width/2, true_values, width, label='True (3-D)', color='#4C72B0')
    bars2 = ax.bar(x + width/2, latent_values, width, label='Latent', color='#55A868')
    for bar in bars1:
        ax.annotate(f'{bar.get_height():.2f}', xy=(bar.get_x() + bar.get_width()/2, bar.get_height()),
                    xytext=(0, 5), textcoords='offset points', ha='center', va='bottom')
    for bar in bars2:
        ax.annotate(f'{bar.get_height():.2f}', xy=(bar.get_x() + bar.get_width()/2, bar.get_height()),
                    xytext=(0, 5), textcoords='offset points', ha='center', va='bottom')
    ax.set_ylabel('Value')
    ax.set_title('Estimated Dimension and Participation Ratio\n(True 3-D vs. Latent)')
    ax.set_xticks(x)
    ax.set_xticklabels(labels)
    ax.legend()
    plt.tight_layout()
    plt.show()
else:
    print(f'Estimated dimension (latent): {est_dim_latent.item():.2f}')
    print(f'Participation ratio (latent): {pr_latent:.2f}')

## 9. Latent Utilisation & S_dim

In [ ]:
z_var = z_all.reshape(-1, N_LATENT).var(dim=0)
p = z_var / z_var.sum()
utilization = -(p * p.log()).sum() / np.log(N_LATENT)
print(f'Latent utilization: {utilization.item():.4f} (1.0 = all dims equally used)')

full_state = values[:5].reshape(-1, values.shape[-1])
latent_flat = z_all[:5].reshape(-1, N_LATENT).numpy()

vars_true = compute_variances(full_state, normalize=True)
vars_latent = compute_variances(latent_flat, normalize=True)

max_len = max(len(vars_true), len(vars_latent))
vars_true_padded = np.pad(vars_true, (0, max_len - len(vars_true)))
vars_latent_padded = np.pad(vars_latent, (0, max_len - len(vars_latent)))

s_dim = compute_s_dim(vars_true_padded, vars_latent_padded)
print(f'S_dim = {s_dim:.4f} (1.0 = perfect variance alignment)')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))

axes[0].bar(range(1, N_LATENT + 1), z_var.numpy())
axes[0].set_xlabel('Latent dimension')
axes[0].set_ylabel('Variance')
axes[0].set_title(f'Latent Variance (utilization={utilization.item():.3f})')

axes[1].plot(range(1, len(vars_true) + 1), vars_true, 'o-', label=f'True state ({len(vars_true)}D)')
axes[1].plot(range(1, len(vars_latent) + 1), vars_latent, 's-', label=f'Latent ({len(vars_latent)}D)')
axes[1].set_xlabel('Component (sorted)')
axes[1].set_ylabel('Normalized variance')
axes[1].set_title(f'Variance Spectrum (S_dim = {s_dim:.3f})')
axes[1].legend()

plt.tight_layout()
plt.show()

## 10. Jacobian Consistency (Velocity Propagation)

Tests whether the learned Jacobian correctly propagates velocity vectors:
$\|(x_{t+2} - x_{t+1}) - e^{\hat{J}\Delta t}(x_{t+1} - x_t)\|^2$

In [ ]:
J_lat_exp = torch.matrix_exp(jacs_full[:, :-2] * dt)

vel_lat = latent_traj_full[:, 1:] - latent_traj_full[:, :-1]
vel_lat_t  = vel_lat[:, :-1]
vel_lat_t1 = vel_lat[:, 1:]

vel_lat_t1_pred = (J_lat_exp @ vel_lat_t.unsqueeze(-1)).squeeze(-1)
residual_lat = vel_lat_t1 - vel_lat_t1_pred
jac_cons_latent = residual_lat.pow(2) / latent_traj_full.var()

print(f'Jacobian consistency (latent, variance-normalized):')
print(f'  mean = {jac_cons_latent.mean().item():.6f}')
print(f'  std  = {jac_cons_latent.mean(dim=(1,2)).std().item():.6f}')

if test_trajs_full is not None and 'true_jacs' in dir():
    J_obs_exp = torch.matrix_exp(true_jacs[:, :-2] * dt)
    obs_ref_raw = (test_trajs_full * sigma + mu)
    vel_obs = obs_ref_raw[:, 1:] - obs_ref_raw[:, :-1]
    vel_obs_t  = vel_obs[:, :-1]
    vel_obs_t1 = vel_obs[:, 1:]
    vel_obs_t1_pred = (J_obs_exp @ vel_obs_t.unsqueeze(-1)).squeeze(-1)
    residual_obs = (vel_obs_t1 - vel_obs_t1_pred) / sigma
    jac_cons_obs = residual_obs.pow(2) / test_trajs_full.var()

    labels_jc = ['obs 3-D (true J)', 'latent (learned J)']
    means_jc = [jac_cons_obs.mean().item(), jac_cons_latent.mean().item()]
    std_errors_jc = [
        jac_cons_obs.flatten().std().item() / (jac_cons_obs.numel() ** 0.5),
        jac_cons_latent.flatten().std().item() / (jac_cons_latent.numel() ** 0.5),
    ]

    fig, ax = plt.subplots(figsize=(6, 4))
    ax.bar(labels_jc, means_jc, yerr=std_errors_jc, capsize=8, color=['C0', 'C1'], alpha=0.7)
    ax.set_ylabel(r'$\|v_{t+1} - e^{J\Delta t} v_t\|^2 / \mathrm{Var}$')
    ax.set_title('Jacobian Consistency (Velocity Propagation)')
    plt.tight_layout()
    plt.show()

## 11. True vs Decoded (No Prediction)

In [ ]:
for i in range(traj_decoded_full.shape[-1]):
    plt.plot(traj_decoded_full[0, :, i].numpy(), c=f'C{i}', linestyle='--', label=f'decoded {i}')

r2_dec = r2_score(
    test_trajs_seq.reshape(-1, test_trajs_seq.shape[-1]),
    traj_decoded_full.reshape(-1, traj_decoded_full.shape[-1]),
)
plt.title(f'Decoded signal (no prediction)\n$R^2 = {r2_dec:.4f}$')
plt.legend()
plt.show()

## 12. False Nearest Neighbors (FNN) Loss

In [ ]:
from JacobianODE.fnn import loss_false

rng_fnn = np.random.default_rng(42)
X_latent = latent_traj_full.reshape(-1, latent_traj_full.shape[-1]).float()
N_POINTS = min(1024, len(X_latent))
idx_latent = rng_fnn.choice(len(X_latent), N_POINTS, replace=False)
X_latent_sub = X_latent[torch.from_numpy(idx_latent)]

with torch.no_grad():
    fnn_loss_latent = loss_false(X_latent_sub, k=2, norm=True).item()

if test_trajs_full is not None:
    X_true = test_trajs_full.reshape(-1, test_trajs_full.shape[-1]).float()
    idx_true = rng_fnn.choice(len(X_true), N_POINTS, replace=False)
    X_true_sub = X_true[torch.from_numpy(idx_true)]
    with torch.no_grad():
        fnn_loss_true = loss_false(X_true_sub, k=2, norm=True).item()
    labels = [f'True state\n(D={X_true.shape[-1]})', f'Latent\n(D={X_latent.shape[-1]})']
    values_fnn = [fnn_loss_true, fnn_loss_latent]
else:
    labels = [f'Latent\n(D={X_latent.shape[-1]})']
    values_fnn = [fnn_loss_latent]

fig, ax = plt.subplots(figsize=(5, 4))
bars = ax.bar(labels, values_fnn, color=['C0', 'C1'][:len(labels)], alpha=0.7)
ax.set_ylabel('FNN loss')
ax.set_title('False Nearest Neighbors Loss')
for bar, val in zip(bars, values_fnn):
    ax.annotate(f'{val:.4f}', xy=(bar.get_x() + bar.get_width()/2, bar.get_height()),
                xytext=(0, 4), textcoords='offset points', ha='center', va='bottom')
plt.tight_layout()
plt.show()

## 13. Noise Amplification Loss

In [ ]:
from JacobianODE.fnn import loss_amplification

N_NEIGHBORS = 2
MAX_T = 5

X_latent_ts = latent_traj_full.float()

if test_trajs_full is not None:
    X_true_ts = test_trajs_full.float()
    with torch.no_grad():
        amp_loss_true = loss_amplification(
            X_true_ts, X_true_ts, n_neighbors=N_NEIGHBORS, max_T=MAX_T, normalize=True
        ).item()
        amp_loss_latent = loss_amplification(
            X_latent_ts, X_true_ts, n_neighbors=N_NEIGHBORS, max_T=MAX_T, normalize=True
        ).item()
    labels = [f'True state\n(D={X_true_ts.shape[-1]})', f'Latent\n(D={X_latent_ts.shape[-1]})']
    values_amp = [amp_loss_true, amp_loss_latent]
else:
    with torch.no_grad():
        amp_loss_latent = loss_amplification(
            X_latent_ts, X_latent_ts, n_neighbors=N_NEIGHBORS, max_T=MAX_T, normalize=True
        ).item()
    labels = [f'Latent\n(D={X_latent_ts.shape[-1]})']
    values_amp = [amp_loss_latent]

fig, ax = plt.subplots(figsize=(5, 4))
bars = ax.bar(labels, values_amp, color=['C0', 'C1'][:len(labels)], alpha=0.7)
ax.set_ylabel(r'Amplification loss $\sigma$')
ax.set_title('Noise Amplification Loss')
for bar, val in zip(bars, values_amp):
    ax.annotate(f'{val:.4f}', xy=(bar.get_x() + bar.get_width()/2, bar.get_height()),
                xytext=(0, 4), textcoords='offset points', ha='center', va='bottom')
plt.tight_layout()
plt.show()